# Model-Free Methods

**Reinforcement Learning Course** — Notebook 03b

We evaluate and control policies **without access to the transition model** $P$.
All algorithms only observe $(s, a, r, s')$ tuples collected by interacting with the Grid World.

| Algorithm | Task | Bootstrap | On/Off-policy |
|-----------|------|-----------|---------------|
| TD(0) | Evaluation | 1 step | — |
| SARSA | Control | 1 step | On |
| Q-Learning | Control | 1 step | Off |
| TD(λ) | Evaluation | λ-weighted | — |

In [ ]:
!pip install numpy plotly

In [ ]:
import sys; sys.path.insert(0, "..")
from practical_case import markov_env as env

## Part 1 — TD(0): evaluating a fixed policy without a model

TD(0) estimates $V^\pi$ by bootstrapping off its own current estimate:

$$V(s) \leftarrow V(s) + \alpha\,\bigl[r(s) + \gamma\, V(s') - V(s)\bigr]$$

We run it on the **spiral CCW** policy and compare snapshots at different episode counts
to $V^\pi$ computed exactly via Bellman iterations (from Notebook 03).

> **Bright cells = high value.** The rightmost panel is the ground truth — watch how the TD estimate converges to it over episodes.

In [ ]:
r      = env.default_reward()   # sparse: +1 at goal ★, 0 elsewhere
gamma  = 0.95
pi_ccw = env.spiral_ccw()

# Ground truth: exact V^pi from iterative Bellman evaluation
V_true = env.policy_eval(pi_ccw, r, gamma)

In [ ]:
# TD(0) snapshots at 4 episode checkpoints
snapshots = env.td_eval(
    pi_ccw, r, gamma,
    n_episodes=3000, alpha=0.05,
    checkpoints=(100, 500, 1500, 3000),
)

env.show_td_snapshots(snapshots, V_true).show()

## Part 2 — Effect of the learning rate $\alpha$

$\alpha$ controls the step size of each update:

- **Small $\alpha$** → slow convergence, smoother curve
- **Large $\alpha$** → fast early progress, noisy or divergent later

We track $\|\hat V - V^\pi\|_\infty$ (sup-norm error) after each episode.
The smoothed curves reveal the long-run trade-off.

In [ ]:
curves = env.td_convergence(
    pi_ccw, r, gamma, V_true,
    alphas=(0.01, 0.05, 0.2),
    n_episodes=3000,
)

env.show_mf_convergence(
    curves,
    ylabel="‖V̂ − Vᵖ‖∞",
    title="TD(0) convergence — effect of α",
    smooth=50,
).show()

## Part 3 — SARSA: on-policy control

SARSA learns $Q^\pi$ by following an $\varepsilon$-greedy policy and updating **with the next action actually taken**:

$$Q(s, a) \leftarrow Q(s, a) + \alpha\,\bigl[r(s) + \gamma\, Q(s', a') - Q(s, a)\bigr]$$

Because $a'$ comes from the same $\varepsilon$-greedy policy, SARSA is **on-policy** — it optimises the policy it is actually executing.

After training we extract the greedy policy $\pi(s) = \arg\max_a Q(s,a)$ and compare it to the optimal policy $\pi^*$ from Policy Iteration.

In [ ]:
Q_sarsa, ret_sarsa = env.sarsa(r, gamma, n_episodes=5000, alpha=0.1, eps=0.1)

V_sarsa  = Q_sarsa.max(axis=1)
pi_sarsa = env.policy_from_q(Q_sarsa)

env.show_value(V_sarsa, pi=pi_sarsa, title="SARSA — learned value & policy").show()

# Reference: optimal policy from Policy Iteration (model-based)
pi_hist = env.policy_iteration(r, gamma=0.95)
V_star, pi_star = pi_hist[-1][1], env.greedy_policy(pi_hist[-1][1], r, gamma)

env.show_value(V_star, pi=pi_star, title="V* — Policy Iteration (reference)").show()

## Part 4 — Q-Learning: off-policy control

Q-Learning bootstraps off the **best possible next action**, regardless of the policy actually followed:

$$Q(s, a) \leftarrow Q(s, a) + \alpha\,\bigl[r(s) + \gamma\, \max_{a'}Q(s', a') - Q(s, a)\bigr]$$

This makes it **off-policy**: the behaviour policy ($\varepsilon$-greedy) explores,
while the target policy is always greedy. Q-Learning converges directly to $Q^*$.

The convergence plot compares **episode returns** (smoothed) for SARSA and Q-Learning over training.
Q-Learning typically rises faster; SARSA is more conservative near dangerous transitions.

In [ ]:
Q_ql, ret_ql = env.q_learning(r, gamma, n_episodes=5000, alpha=0.1, eps=0.1)

V_ql  = Q_ql.max(axis=1)
pi_ql = env.policy_from_q(Q_ql)

env.show_value(V_ql, pi=pi_ql, title="Q-Learning — learned value & policy").show()

# Compare episode returns: SARSA vs Q-Learning
env.show_mf_convergence(
    {"SARSA": ret_sarsa, "Q-Learning": ret_ql},
    ylabel="episode return",
    title="Episode returns — SARSA vs Q-Learning (smoothed)",
    smooth=100,
).show()

## Part 5 — (Optional) TD(λ): bridging TD and Monte Carlo

TD(λ) uses **eligibility traces** $e(s)$ to credit past states for the current TD error:

$$e(s) \leftarrow \gamma\lambda\, e(s) + \mathbf{1}[S_t = s]$$
$$V(s) \leftarrow V(s) + \alpha\,\delta_t\, e(s)$$

- $\lambda = 0$ → pure TD(0): one-step bootstrap only
- $\lambda = 1$ → Monte Carlo: full-episode return (no discount on traces)
- $0 < \lambda < 1$ → interpolates between them

The plot shows how $\lambda$ affects convergence speed on the same policy evaluation task.

In [ ]:
curves_lam = env.td_lambda_convergence(
    pi_ccw, r, gamma, V_true,
    lambdas=(0.0, 0.5, 0.9),
    n_episodes=3000, alpha=0.05,
)

env.show_mf_convergence(
    curves_lam,
    ylabel="‖V̂ − Vᵖ‖∞",
    title="TD(λ) convergence — effect of λ",
    smooth=50,
).show()